# Sistema de Reconocimiento de Voz — Speech Commands

**Proyecto escolar — Procesamiento de Voz**

Arquitectura implementada:
1. **Extracción de características**: MFCC, Log-Mel, LPC, PLP (implementados paso a paso)
2. **Métodos clásicos**: DTW, K-Means, GMM, HMM
3. **Modelo End-to-End**: Bi-LSTM + CTC Loss
4. **Decodificación**: Greedy Search y Beam Search con lexicón

**Entorno**: `conda activate K-means-clustering`

## 0. Configuración e importaciones

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pickle
import numpy as np
import matplotlib.pyplot as plt
import kagglehub

from src.config import TARGET_WORDS, SAMPLES_PER_CLASS, CTC_MODEL_PATH
from src.dataset import collect_samples, find_dataset_root, train_val_split, extract_mfcc_file
from src.features.mfcc import compute_mfcc
from src.features.log_mel import compute_log_mel
from src.features.lpc import compute_lpc
from src.features.plp import compute_plp
from src.classic.dtw import dtw_predict
from src.classic.kmeans_classifier import KMeansWordClassifier
from src.classic.gmm_classifier import GMMWordClassifier
from src.classic.hmm_classifier import SimpleHMMClassifier
from src.train_ctc import train_ctc_model, load_ctc_model
from src.predict_ctc import predict_greedy, match_to_vocabulary, word_accuracy
from src.predict_beam_search import predict_beam_search
from src.metrics import accuracy, confusion_matrix, classification_report

print('Palabras objetivo:', TARGET_WORDS)
print('Proyecto:', PROJECT_ROOT)

## 1. Descarga del dataset (Kaggle Speech Commands)

In [ ]:
# Descarga con kagglehub (requiere credenciales Kaggle configuradas)
dataset_path = kagglehub.dataset_download('yashdogra/speech-commands')
print('Dataset descargado en:', dataset_path)

root = find_dataset_root(dataset_path)
print('Raíz del dataset:', root)

samples = collect_samples(root, max_per_class=SAMPLES_PER_CLASS)
train_samples, val_samples = train_val_split(samples)
print(f'Muestras: {len(samples)} total | {len(train_samples)} train | {len(val_samples)} val')

## 2. Extracción de características acústicas

### 2.1 MFCC — Mel-Frequency Cepstral Coefficients
Pipeline: preénfasis → ventaneo → FFT → banco Mel → log → DCT

In [ ]:
example_path, example_label = train_samples[0]
from src.audio_io import load_wav, normalize_audio

audio, sr = load_wav(example_path)
audio = normalize_audio(audio)

mfcc = compute_mfcc(audio, sample_rate=sr)
log_mel = compute_log_mel(audio, sample_rate=sr)
lpc = compute_lpc(audio)
plp = compute_plp(audio, sample_rate=sr)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0,0].plot(audio); axes[0,0].set_title(f'Forma de onda — "{example_label}"')
axes[0,1].imshow(mfcc.T, aspect='auto', origin='lower', cmap='magma')
axes[0,1].set_title('MFCC (13 coef.)'); axes[0,1].set_ylabel('Coeficiente')
axes[1,0].imshow(log_mel.T, aspect='auto', origin='lower', cmap='viridis')
axes[1,0].set_title('Log-Mel Filterbanks')
axes[1,1].imshow(plp.T, aspect='auto', origin='lower', cmap='plasma')
axes[1,1].set_title('PLP (Perceptual Linear Prediction)')
plt.tight_layout(); plt.show()
print(f'MFCC shape: {mfcc.shape} | LPC shape: {lpc.shape}')

## 3. Métodos clásicos de reconocimiento

Comparamos DTW, K-Means, GMM y HMM sobre el conjunto de validación.

In [ ]:
print('Extrayendo MFCC para entrenamiento clásico...')
train_mfcc = [extract_mfcc_file(p) for p, _ in train_samples]
train_labels = [l for _, l in train_samples]
val_paths = [p for p, _ in val_samples]
val_labels = [l for _, l in val_samples]
val_mfcc = [extract_mfcc_file(p) for p in val_paths]

# --- K-Means ---
kmeans_clf = KMeansWordClassifier(n_clusters=len(TARGET_WORDS))
kmeans_clf.fit(train_mfcc, train_labels)
kmeans_preds = [kmeans_clf.predict(f) for f in val_mfcc]
kmeans_acc = accuracy(val_labels, kmeans_preds)

# --- GMM ---
gmm_clf = GMMWordClassifier(n_components=8)
gmm_clf.fit(train_mfcc, train_labels)
gmm_preds = [gmm_clf.predict(f) for f in val_mfcc]
gmm_acc = accuracy(val_labels, gmm_preds)

# --- HMM ---
hmm_clf = SimpleHMMClassifier(n_symbols=64, n_states=5)
hmm_clf.fit(train_mfcc, train_labels)
hmm_preds = [hmm_clf.predict(f) for f in val_mfcc]
hmm_acc = accuracy(val_labels, hmm_preds)

# --- DTW (plantillas: 5 por palabra) ---
templates = {}
for word in TARGET_WORDS:
    templates[word] = [f for f, l in zip(train_mfcc, train_labels) if l == word][:5]
dtw_preds = [dtw_predict(f, templates) for f in val_mfcc]
dtw_acc = accuracy(val_labels, dtw_preds)

print('\n=== Resultados métodos clásicos ===')
print(f'K-Means:  {kmeans_acc:.2%}')
print(f'GMM:      {gmm_acc:.2%}')
print(f'HMM:      {hmm_acc:.2%}')
print(f'DTW:      {dtw_acc:.2%}')

In [ ]:
# Gráfica comparativa
methods = ['K-Means', 'GMM', 'HMM', 'DTW']
accs = [kmeans_acc, gmm_acc, hmm_acc, dtw_acc]
colors = ['#38bdf8', '#4ade80', '#fbbf24', '#f472b6']

plt.figure(figsize=(8, 5))
bars = plt.bar(methods, accs, color=colors)
plt.ylim(0, 1); plt.ylabel('Exactitud'); plt.title('Comparación métodos clásicos')
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{acc:.1%}', ha='center')
plt.tight_layout(); plt.show()

# Matriz de confusión K-Means
cm = confusion_matrix(val_labels, kmeans_preds, TARGET_WORDS)
plt.figure(figsize=(8, 6))
plt.imshow(cm, cmap='Blues')
plt.colorbar(); plt.xticks(range(len(TARGET_WORDS)), TARGET_WORDS, rotation=45)
plt.yticks(range(len(TARGET_WORDS)), TARGET_WORDS)
plt.xlabel('Predicción'); plt.ylabel('Real'); plt.title('Matriz de confusión — K-Means')
plt.tight_layout(); plt.show()

## 4. Modelo End-to-End: Bi-LSTM + CTC

**Fase de entrenamiento** (según diagrama del proyecto):
- Entrada: MFCC
- Modelo acústico: Bi-LSTM
- Función de pérdida: CTC (no requiere alineación frame-a-frame)

In [ ]:
train_paths = [p for p, _ in train_samples]
train_lbls = [l for _, l in train_samples]

print('Entrenando Bi-LSTM + CTC (puede tardar varios minutos)...')
model = train_ctc_model(
    train_paths, train_lbls,
    val_paths=val_paths, val_labels=val_labels,
    epochs=10,
)
print('Modelo guardado en:', CTC_MODEL_PATH)

In [ ]:
# Curva de pérdida
checkpoint = __import__('torch').load(CTC_MODEL_PATH, map_location='cpu', weights_only=False)
history = checkpoint['history']

plt.figure(figsize=(8, 4))
plt.plot(history['train_loss'], label='Train Loss', marker='o')
if history.get('val_loss'):
    plt.plot(history['val_loss'], label='Val Loss', marker='s')
plt.xlabel('Época'); plt.ylabel('CTC Loss'); plt.title('Curva de entrenamiento Bi-LSTM + CTC')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 5. Fase de inferencia — Greedy Search vs Beam Search

In [ ]:
model, _ = load_ctc_model(CTC_MODEL_PATH)

greedy_preds = [match_to_vocabulary(predict_greedy(model, p)) for p in val_paths]
beam_preds = [predict_beam_search(model, p, beam_width=10) for p in val_paths]

greedy_acc = word_accuracy(greedy_preds, val_labels)
beam_acc = word_accuracy(beam_preds, val_labels)

print('=== Resultados CTC ===')
print(f'Greedy Search: {greedy_acc:.2%}')
print(f'Beam Search:   {beam_acc:.2%}')

report = classification_report(val_labels, greedy_preds, TARGET_WORDS)
print('\nReporte por clase (Greedy):')
for word, m in report.items():
    print(f'  {word:6s} — P:{m["precision"]:.2f} R:{m["recall"]:.2f} F1:{m["f1"]:.2f}')

In [ ]:
# Comparación final de todos los métodos
all_methods = methods + ['CTC Greedy', 'CTC Beam']
all_accs = accs + [greedy_acc, beam_acc]

plt.figure(figsize=(10, 5))
plt.bar(all_methods, all_accs, color=['#38bdf8','#4ade80','#fbbf24','#f472b6','#a78bfa','#fb923c'])
plt.ylim(0, 1); plt.ylabel('Exactitud'); plt.title('Comparación completa de métodos ASR')
plt.xticks(rotation=30, ha='right')
for i, acc in enumerate(all_accs):
    plt.text(i, acc + 0.02, f'{acc:.1%}', ha='center')
plt.tight_layout(); plt.show()

## 6. Guardar modelos clásicos para la interfaz web

In [ ]:
from src.config import KMEANS_MODEL_PATH, GMM_MODEL_PATH, DTW_TEMPLATES_PATH

kmeans_clf.save(KMEANS_MODEL_PATH)
gmm_clf.save(GMM_MODEL_PATH)
DTW_TEMPLATES_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(DTW_TEMPLATES_PATH, 'wb') as f:
    pickle.dump(templates, f)

print('Modelos guardados. Para probar la interfaz web ejecuta:')
print('  python app/web_app.py')
print('  Abre http://127.0.0.1:5000')

## 7. Prueba con un archivo individual

In [1]:
test_path, test_label = val_samples[0]
print(f'Archivo: {test_path}')
print(f'Etiqueta real: {test_label}')
print(f'CTC Greedy: {match_to_vocabulary(predict_greedy(model, test_path))}')
print(f'CTC Beam:   {predict_beam_search(model, test_path)}')
print(f'K-Means:    {kmeans_clf.predict(extract_mfcc_file(test_path))}')
print(f'GMM:        {gmm_clf.predict(extract_mfcc_file(test_path))}')

NameError: name 'val_samples' is not defined